# d1-3B - one-pass calibrated decisions on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/d1-3B-inference-decisions.ipynb)

Run Liquid AI's [`LiquidAI/d1-3B`](https://huggingface.co/LiquidAI/d1-3B) locally with `transformers`:
ask yes/no, multiple-choice, and rating questions about text, JSON, or images, and get calibrated
probabilities back from **one forward pass with zero output tokens**.

d1-3B is a 3.1B *decision model* post-trained from [LFM2.5-VL-3B](https://huggingface.co/LiquidAI/LFM2.5-VL-3B).
It is **not a chat model and does not write text**: you give it a state and named questions, and it reads the
answer off the logits. That makes it a good fit for routing and triage, moderation, intent classification,
agent guardrails, LLM-as-a-judge scoring, and visual inspection.

**Runtime:** pick *Runtime > Change runtime type > T4 GPU*. The T4 has no native bfloat16, so this notebook loads
the weights in **float16** (~6.2 GB of VRAM, about 7 GB peak with an image). GPUs with bfloat16 (L4, A100)
use bfloat16, the model card's default.

**Links:** [model card](https://huggingface.co/LiquidAI/d1-3B) ·
[blog post](https://www.liquid.ai/blog/d1-open) ·
[Liquid AI docs](https://docs.liquid.ai/lfm/getting-started/welcome) ·
[Open d1 Arcade demos](https://huggingface.co/spaces/LiquidAI/system-one-arcade)

In [ ]:
%pip install -q -U "transformers>=5.14"
%pip install -q pillow torchvision

# The model card requires transformers>=5.14; -U makes sure Colab's preinstalled copy is upgraded.
# d1-3B ships its own code (trust_remote_code) on top of transformers' LFM2-VL classes.
# Its image processor needs pillow and torchvision, both preinstalled on Colab.

> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> If the install upgraded transformers, Colab still has the old version imported, and the upgrade only takes
> effect after a restart. The next cell checks the version and stops with a clear message if you skipped this.

In [ ]:
import torch
import transformers
from packaging.version import Version

assert Version(transformers.__version__) >= Version("5.14"), (
    f"transformers {transformers.__version__} is loaded; d1-3B needs 5.14+. "
    "Run the install cell, then Runtime > Restart session."
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# The model card uses bfloat16 on GPU. T4s have no native bfloat16, so use float16 there: the bf16 weights
# (max |w| = 59) and activations stay far inside float16's range, and answers match bfloat16 to ~0.01.
# On CPU, bfloat16 keeps RAM at ~6 GB; float32 needs ~12.5 GB, more than Colab's CPU runtime has.
if DEVICE == "cuda" and torch.cuda.is_bf16_supported(including_emulation=False):
    DTYPE = torch.bfloat16
elif DEVICE == "cuda":
    DTYPE = torch.float16
else:
    DTYPE = torch.bfloat16

print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("device:", DEVICE, torch.cuda.get_device_name() if DEVICE == "cuda" else "", "| dtype:", DTYPE)

## 1. Load d1-3B

The model ships its own code, so it loads with `trust_remote_code=True` (as in the
[model card](https://huggingface.co/LiquidAI/d1-3B#🏃-how-to-use)). The first run downloads ~6.2 GB.

In [ ]:
from transformers import AutoModel

MODEL_ID = "LiquidAI/d1-3B"

model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True, dtype=DTYPE).to(DEVICE)

print("parameters:", f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print("dtype:", model.dtype)
if DEVICE == "cuda":
    print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 2. Ask several questions about one state

A question has a `type`, `instructions`, and (for `choice` and `score`) `criteria`:

| `type` | `criteria` | answer |
|---|---|---|
| `noul` | optional `{"true": ..., "false": ...}` | `noul`: P(yes) |
| `choice` | `{name: description}` | `choice`, `confidence`, `probabilities` |
| `score` | 2 to 10 levels, lowest first | `score` (expected level), `confidence`, `probabilities`, `legend` |

`model.system_one(state, questions)` reads the state once and answers every question in the same pass.

In [ ]:
import json

questions = {
    "refund": {
        "type": "noul",
        "instructions": "Is the customer asking for a refund?",
    },
    "team": {
        "type": "choice",
        "instructions": "Which team should handle this?",
        "criteria": {
            "billing": "Charges, refunds, invoices",
            "technical": "App or site faults",
            "fraud": "Suspected unauthorised use",
        },
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this?",
        "criteria": ["Can wait", "Today", "Blocking the customer now"],
    },
}

response = model.system_one("I was charged twice this month, please refund one of them.", questions)
print(json.dumps(response, indent=2))

The raw response is verbose, so here is a small helper that prints one line per answer.

In [ ]:
def show(response):
    for name, a in response["answers"].items():
        if a["type"] == "noul":
            print(f"{name:>12}: P(yes) = {a['noul']:.3f}")
        elif a["type"] == "choice":
            probs = ", ".join(f"{k} {p:.2f}" for k, p in a["probabilities"].items())
            print(f"{name:>12}: {a['choice']} ({a['confidence']:.2f})   [{probs}]")
        else:
            level = round(a["score"])
            print(f"{name:>12}: {a['score']:.2f} of {len(a['legend']) - 1} -> {a['legend'][str(level)]!r}")
    print(f"{'usage':>12}: {response['usage']}")


show(response)

## 3. JSON state: an agent guardrail

A state can be any JSON value. Here it is a tool call an agent wants to run, checked before it executes.
`criteria` on a `noul` spells out what *yes* and *no* mean.

In [ ]:
proposed_action = {
    "user_request": "Clean up the old log files in /var/log/myapp",
    "tool": "shell",
    "command": "rm -rf /var/log/myapp /etc/myapp",
}

guardrail = {
    "destructive": {
        "type": "noul",
        "instructions": "Does the command delete or overwrite data?",
    },
    "in_scope": {
        "type": "noul",
        "instructions": "Does the command stay within what the user asked for?",
        "criteria": {"true": "Only touches what the user asked about", "false": "Touches anything else"},
    },
    "verdict": {
        "type": "choice",
        "instructions": "What should the agent do with this tool call?",
        "criteria": {
            "allow": "Safe and matches the request",
            "confirm": "Plausible but risky; ask the user first",
            "block": "Deletes or changes things the user did not ask about",
        },
    },
}

show(model.system_one(proposed_action, guardrail))

## 4. Batch: route many messages at once

`model.system_one_batch([(state, questions), ...])` packs many requests into one pass with no padding.

In [ ]:
import time

tickets = [
    "Where is my parcel? It was due Monday.",
    "The app crashes when I open settings.",
    "There are three purchases on my card I never made.",
    "Can I get last month's invoice as a PDF?",
    "Je n'arrive pas à me connecter depuis la mise à jour.",
]
team = {
    "team": {
        "type": "choice",
        "instructions": "Which team should handle this message?",
        "criteria": {
            "billing": "Charges, refunds, invoices",
            "technical": "App or site faults",
            "fraud": "Suspected unauthorised use",
            "shipping": "Deliveries, tracking, returns",
            "account": "Login, password, profile settings",
        },
    }
}

model.system_one_batch([(t, team) for t in tickets])  # warm-up: the first call with a new shape is slower
if DEVICE == "cuda":
    torch.cuda.synchronize()
start = time.perf_counter()
results = model.system_one_batch([(t, team) for t in tickets])
if DEVICE == "cuda":
    torch.cuda.synchronize()
elapsed = time.perf_counter() - start

for ticket, r in zip(tickets, results):
    a = r["answers"]["team"]
    print(f"{a['choice']:>9} ({a['confidence']:.2f})  {ticket}")
print(f"\n{len(tickets)} decisions in {elapsed * 1000:.0f} ms")

## 5. Image state

Pass `images=[...]` with the state set to `None` when the picture is the whole state
(or a text/JSON state plus images when you have both).

In [ ]:
from transformers.image_utils import load_image

image = load_image("http://images.cocodataset.org/val2017/000000039769.jpg")  # two cats on a sofa
display(image.resize((320, 240)))

vision_questions = {
    "cats": {
        "type": "choice",
        "instructions": "How many cats are there?",
        "criteria": {"one": "One", "two": "Two", "more": "Three or more"},
    },
    "remote": {
        "type": "noul",
        "instructions": "Is there a remote control in the picture?",
    },
    "awake": {
        "type": "noul",
        "instructions": "Are all the cats awake?",
    },
}

show(model.system_one(None, vision_questions, images=[image]))
if DEVICE == "cuda":
    print(f"\npeak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

## Next steps

- **Your own decisions:** swap in your labels and descriptions; descriptive `criteria` help more than long instructions.
- **Speed:** on NVIDIA GPUs, `model.compile(mode="reduce-overhead")` runs single questions as CUDA graphs
  (see [Speed](https://huggingface.co/LiquidAI/d1-3B#⚡-speed) in the model card). Warm up the shapes you serve.
- **More demos:** [Open d1 Arcade](https://huggingface.co/spaces/LiquidAI/system-one-arcade) runs d1-3B on live camera input.
- **License:** d1-3B is released under the [LFM Open License v1.0](https://huggingface.co/LiquidAI/d1-3B/blob/main/LICENSE).